# 05 — Legs & Formation Classification

## Why legs?

A valid Supply & Demand zone is not just a tight cluster — it is a **balanced structure with a price memory**.
The leg-in carries the energy that *created* the base; the leg-out is the burst that *leaves* it.
Without measuring both we cannot tell whether a cluster was formed by institutional intent or by random noise.

## The four formations

| Leg-in | Leg-out | Code | Zone type | Pattern name |
|--------|---------|------|-----------|--------------|
| Up     | Up      | RBR  | Demand    | Rally-Base-Rally |
| Down   | Down    | DBD  | Supply    | Drop-Base-Drop |
| Down   | Up      | DBR  | Demand    | Drop-Base-Rally |
| Up     | Down    | RBD  | Supply    | Rally-Base-Drop |

If either leg is **flat** the cluster is discarded — both momentum signals must be present.

## Leg-strength formula

Given cluster `[bs, be]` and window size $L$ = `LEG_CANDLES`:

$$
\text{leg\_in\_net}  = \text{close}_{bs-1} - \text{open}_{bs-L}
$$

$$
\text{leg\_out\_net} = \text{close}_{be+L} - \text{close}_{be}
$$

A leg is **directional** if $|\text{net}| \ge \text{LEG\_ATR\_MIN} \times \overline{\text{ATR}}_{\text{base}}$, otherwise it is **flat**.

| Constant | Default | Meaning |
|---|---|---|
| `LEG_CANDLES` | 3 | Bars to look back/forward on each side |
| `LEG_ATR_MIN` | 1.5 | Minimum ATR-multiples for a directional leg |

## 1. Setup & load data

In [13]:
import sys
sys.path.append("..")

import pandas as pd
from IPython.display import display

from utils.data_loader import load_all_timeframes
from utils.models import CandlePrimitives, add_atr
from utils.base_detector import detect_bases
from utils.legs_formation import detect_formations, FORMATION_MAP
from utils.config import (
    COLOR_BULL, COLOR_BEAR, COLOR_BASE,
    CHART_BG as BG, CHART_GRID as GRID,
    LEG_CANDLES, LEG_ATR_MIN,
)

SYMBOL = "USDJPY=X"

raw  = load_all_timeframes(SYMBOL, align=True)
data = {tf: add_atr(CandlePrimitives.enrich_dataframe(df)) for tf, df in raw.items()}

print(f"Loaded {len(data)} timeframes: {list(data.keys())}")
for tf, df in data.items():
    print(f"  {tf:4s}  {len(df):5d} rows  {df.index[0].date()} → {df.index[-1].date()}")

Loaded 4 timeframes: ['1wk', '1d', '4h', '1h']
  1wk     104 rows  2024-06-09 → 2026-05-31
  1d      517 rows  2024-06-04 → 2026-06-03
  4h     3175 rows  2024-06-04 → 2026-06-04
  1h    12262 rows  2024-06-04 → 2026-06-04


## 2. Detect bases, then classify formations

In [14]:
# Run base detection + formation classification on all timeframes
results = {}
for tf, df in data.items():
    passed, _ = detect_bases(df)
    formations = detect_formations(df, passed)
    results[tf] = {
        "df":         df,
        "passed":     passed,
        "formations": formations,
    }

# Summary table
rows = []
for tf, r in results.items():
    fms = r["formations"]
    n_demand = sum(1 for f in fms if f["zone_type"] == "demand")
    n_supply = sum(1 for f in fms if f["zone_type"] == "supply")
    n_bases  = len(r["passed"])
    n_skipped = n_bases - len(fms)
    rows.append({
        "timeframe":          tf,
        "passed_bases":       n_bases,
        "formations_found":   len(fms),
        "demand_zones":       n_demand,
        "supply_zones":       n_supply,
        "skipped (flat leg)": n_skipped,
        "form_rate":          f"{len(fms)/n_bases:.0%}" if n_bases else "—",
    })

summary = pd.DataFrame(rows).set_index("timeframe")
display(summary)

,passed_bases,formations_found,demand_zones,supply_zones,skipped (flat leg),form_rate
timeframe,,,,,,
1wk,25,2,1,1,23,8%
1d,72,1,0,1,71,1%
4h,812,33,15,18,779,4%
1h,3108,135,68,67,2973,4%


## 3. Inspect — daily formations

In [15]:
TF = "1d"
df_1d = results[TF]["df"]

detail_rows = []
for f in results[TF]["formations"]:
    bs, be = f["start"], f["end"]
    detail_rows.append({
        "date_start":   df_1d.index[bs].date(),
        "date_end":     df_1d.index[be].date(),
        "base_candles": f["count"],
        "formation":    f["formation"],
        "zone_type":    f["zone_type"],
        "leg_in_dir":   f["leg_in_dir"],
        "leg_out_dir":  f["leg_out_dir"],
        "leg_in_net":   f["leg_in_net"],
        "leg_out_net":  f["leg_out_net"],
        "avg_atr":      f["avg_atr"],
    })

detail = pd.DataFrame(detail_rows)
if detail.empty:
    print(f"No formations found for {TF}")
else:
    display(detail.to_string(index=False))

'date_start   date_end  base_candles formation zone_type leg_in_dir leg_out_dir  leg_in_net  leg_out_net  avg_atr\n2026-02-04 2026-02-10             5       RBD    supply         up        down     2.29599       -3.311  1.48306'

## 4. Visualize — all 4 timeframes

In [16]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import timedelta

PLOT_MONTHS = 2

# Color by zone type
FILL = {
    "demand": "rgba(38,166,154,0.18)",
    "supply": "rgba(239,83,80,0.18)",
}
LINE_COLOR = {
    "demand": "#26a69a",
    "supply": "#ef5350",
}

def plot_formations(tf: str, r: dict) -> None:
    df          = r["df"]
    formations  = r["formations"]

    # Slice to last PLOT_MONTHS
    cutoff   = df.index[-1] - pd.DateOffset(months=PLOT_MONTHS)
    df_plot  = df[df.index >= cutoff].copy()
    plot_start = df_plot.index[0]
    offset   = len(df) - len(df_plot)

    # Only show formations whose cluster end is inside the window
    vis_fms = [f for f in formations if f["end"] >= offset]

    n_demand_all = sum(1 for f in formations if f["zone_type"] == "demand")
    n_supply_all = sum(1 for f in formations if f["zone_type"] == "supply")

    fig = go.Figure()

    # Candlestick
    fig.add_trace(go.Candlestick(
        x=df_plot.index,
        open=df_plot["open"],
        high=df_plot["high"],
        low=df_plot["low"],
        close=df_plot["close"],
        increasing_line_color=COLOR_BULL,
        decreasing_line_color=COLOR_BEAR,
        name="Price",
    ))

    # Shade each formation's base cluster
    for f in vis_fms:
        bs_plot = f["start"] - offset
        be_plot = f["end"]   - offset

        # Clamp to the visible window
        bs_plot = max(bs_plot, 0)
        be_plot = min(be_plot, len(df_plot) - 1)

        x0 = df_plot.index[bs_plot]
        x1 = df_plot.index[be_plot]
        zt = f["zone_type"]

        # Vertical region for the base
        fig.add_vrect(
            x0=x0, x1=x1,
            fillcolor=FILL[zt],
            line_color=LINE_COLOR[zt],
            line_width=1,
            opacity=1.0,
        )

        # Formation label centered on the cluster
        mid_pos = (bs_plot + be_plot) // 2
        mid_x   = df_plot.index[mid_pos]
        mid_y   = df_plot["high"].iloc[bs_plot : be_plot + 1].max()
        fig.add_annotation(
            x=mid_x, y=mid_y,
            text=f["formation"],
            showarrow=False,
            font=dict(size=10, color=LINE_COLOR[zt]),
            yanchor="bottom",
        )

    fig.update_layout(
        title=(
            f"{SYMBOL} {tf} — Formations in last {PLOT_MONTHS} months "
            f"(shown: {len(vis_fms)} | full history: "
            f"{n_demand_all}D + {n_supply_all}S = {len(formations)} total)"
        ),
        xaxis_rangeslider_visible=False,
        template="plotly_dark",
        paper_bgcolor=BG,
        plot_bgcolor=BG,
        xaxis=dict(gridcolor=GRID),
        yaxis=dict(gridcolor=GRID),
        height=480,
    )
    fig.show()


for tf, r in results.items():
    plot_formations(tf, r)

## 5. What's next

| Notebook | Topic | Status |
|---|---|---|
| `06_proximal_distal_departure.ipynb` | Zone boundaries (proximal/distal) + departure strength | ⏳ next |
| `07_verify_all_scenarios.ipynb` | End-to-end pipeline — all filters together | ⏳ |

With formations classified we now know:
- **Which side** of the market each zone represents (demand vs supply)
- **How strong** the move was that created and left the base

Next we define the precise **entry / reaction levels** (proximal line, distal line) and verify that price **departed** fast enough to confirm institutional intent.

# 05 — Legs & Formation Classification

## Goal
Every valid S&D zone has three parts: **Leg-In → Base → Leg-Out**.  
This notebook measures those legs and classifies each cluster into one of four formation types.

## What Is a "Leg"?
A leg is a **directional move** into or out of the base, measured by net price displacement.

$$\text{leg\_net} = \text{close}_{n} - \text{open}_{1}$$

where candles $1 \dots n$ are the consecutive non-base candles immediately before or after the cluster.

## Leg Strength Requirement
A leg must be **strong enough** to count:

$$\frac{|\text{leg\_net}|}{\text{avg\_ATR}} \geq 1.5$$

A weak leg means price was drifting, not impulsively moving — no zone is formed.

## Formation Types

Each zone is classified by the direction of Leg-In and Leg-Out:

| Formation | Leg-In Direction | Leg-Out Direction | Zone Type |
|-----------|-----------------|-------------------|-----------|
| **RBR** — Rally-Base-Rally | Up | Up | Demand |
| **DBD** — Drop-Base-Drop | Down | Down | Supply |
| **DBR** — Drop-Base-Rally | Down | Up | Demand |
| **RBD** — Rally-Base-Drop | Up | Down | Supply |

A "flat" or absent leg → no formation → no zone.

## Visualization
The chart shows clusters colored by formation type, with the leg-in and leg-out arrows annotated.


## 1. Constants and data
Two groups of thresholds:
- The **base-cluster constants** must match NB04 exactly. We re-declare them here so the notebook can run standalone.
- The **leg constants** are new: `LEG_CANDLES` is the lookback/lookforward window size, and `LEG_ATR_MIN` is the strength requirement.

We pull `open / high / low / close / atr` into NumPy arrays for speed and to make the indexing in `measure_legs()` read naturally.


In [17]:
import pandas as pd
import numpy as np

# base-cluster constants (must match NB04)
BASE_BODY_RATIO_MAX  = 0.50
BASE_MIN_CANDLES     = 1
BASE_MAX_CANDLES     = 5
BASE_MAX_ATR_WIDTH   = 2.5
BASE_COMPACTNESS_MAX = 0.80

# new: leg constants
LEG_CANDLES = 3      # window size on each side of the base
LEG_ATR_MIN = 1.5    # leg must move at least 1.5x avg ATR to count

df      = pd.read_csv("../fixtures_enhanced.csv", index_col=0, parse_dates=True)
labeled = pd.read_csv("../fixtures_labeled.csv",  index_col=0, parse_dates=True)

o   = df["open"].values
h   = df["high"].values
l   = df["low"].values
c   = df["close"].values
atr = df["atr"].values

print(f"{len(df)} candles loaded")


98 candles loaded


## 2. The formation map
Each formation is just a `(leg_in_direction, leg_out_direction)` pair. Encoding it as a dict means the classifier is a one-line lookup later — no nested ifs, no order-of-operations bugs.

| Leg-In | Leg-Out | Formation | Zone Type |
|--------|---------|-----------|-----------|
| up     | up      | **RBR** — Rally-Base-Rally  | demand |
| down   | down    | **DBD** — Drop-Base-Drop    | supply |
| down   | up      | **DBR** — Drop-Base-Rally   | demand |
| up     | down    | **RBD** — Rally-Base-Drop   | supply |

Anything else (e.g. `flat` on either side) means there's no real formation and the cluster is discarded.


In [18]:
FORMATION_MAP = {
    ("up",   "up"):   ("RBR", "demand"),
    ("down", "down"): ("DBD", "supply"),
    ("down", "up"):   ("DBR", "demand"),
    ("up",   "down"): ("RBD", "supply"),
}

print("Formation map:")
for (lin, lout), (form, zt) in FORMATION_MAP.items():
    print(f"  leg-in={lin:5s}  leg-out={lout:5s}  →  {form}  ({zt})")


Formation map:
  leg-in=up     leg-out=up     →  RBR  (demand)
  leg-in=down   leg-out=down   →  DBD  (supply)
  leg-in=down   leg-out=up     →  DBR  (demand)
  leg-in=up     leg-out=down   →  RBD  (supply)


## 3. Base-cluster helpers (same logic as NB04)
We re-implement `find_base_clusters` and `cluster_ok` directly against NumPy arrays so this notebook is self-contained.

`cluster_ok` applies the same two tightness gates from NB04:

1. **Width / ATR** — `(base_high − base_low) / ATR ≤ 2.5`
2. **Compactness** — `(close_max − close_min) / base_width ≤ 0.80`


In [19]:
def find_base_clusters():
    clusters = []
    i = 0
    while i < len(df):
        if not df["is_base"].iloc[i]:
            i += 1
            continue
        j = i
        while j + 1 < len(df) and df["is_base"].iloc[j + 1]:
            j += 1
        if BASE_MIN_CANDLES <= (j - i + 1) <= BASE_MAX_CANDLES:
            clusters.append((i, j))
        i = j + 1
    return clusters


def cluster_ok(bs, be):
    avg_atr = atr[bs : be + 1].mean()
    width   = h[bs : be + 1].max() - l[bs : be + 1].min()
    compact = (c[bs : be + 1].max() - c[bs : be + 1].min()) / width if width > 0 else 1
    return (width <= BASE_MAX_ATR_WIDTH * avg_atr) and (compact <= BASE_COMPACTNESS_MAX)


raw_clusters = find_base_clusters()
ok_clusters  = [(bs, be) for bs, be in raw_clusters if cluster_ok(bs, be)]
print(f"{len(raw_clusters)} raw clusters  →  {len(ok_clusters)} pass tightness gates")


21 raw clusters  →  20 pass tightness gates


## 4. `classify_move` — turn a net displacement into a direction
Given the net price change of a leg and the local ATR, decide whether the move is `"up"`, `"down"`, or `"flat"`. The threshold scales with volatility — a 1-point move during quiet times is huge, but trivial when ATR is 5.

$$\text{direction} = \begin{cases} \text{up}   & \text{net} \geq +1.5 \cdot \overline{\text{ATR}} \\ \text{down} & \text{net} \leq -1.5 \cdot \overline{\text{ATR}} \\ \text{flat} & \text{otherwise} \end{cases}$$


In [20]:
def classify_move(net, avg_atr):
    threshold = LEG_ATR_MIN * avg_atr
    if net >= threshold:
        return "up"
    if net <= -threshold:
        return "down"
    return "flat"


## 5. `measure_legs` — the core windowing logic
Given a cluster `[bs, be]`, look `LEG_CANDLES` bars back for the leg-in and `LEG_CANDLES` bars forward for the leg-out.

```
   ┌─── leg-in window ───┐  ┌── base ──┐  ┌── leg-out window ──┐
   bs-3   bs-2   bs-1    │  bs ... be   │  be+1   be+2   be+3
                         │              │
                  start from open[bs-3]    end at close[be+3]
                  end   at close[bs-1]
```

Net displacements:

$$\text{leg\_in\_net}  = \text{close}_{bs-1} - \text{open}_{bs - L}$$
$$\text{leg\_out\_net} = \text{close}_{be + L} - \text{close}_{be}$$

where $L$ = `LEG_CANDLES`.

**Guard rails (critical)**:
- If `bs < LEG_CANDLES`, there isn't enough history — Python would happily index with `-1`, wrapping around to the *end* of the array (the famous look-ahead bug). We return `None`.
- If `be + LEG_CANDLES >= len(c)`, there isn't enough future — same story, return `None`.

These two guards are the cleanest way to avoid silent data corruption.


In [21]:
def measure_legs(bs, be):
    avg_atr = atr[bs : be + 1].mean()

    if bs < LEG_CANDLES:
        return None                         # not enough history
    leg_in_net = c[bs - 1] - o[bs - LEG_CANDLES]

    if be + LEG_CANDLES >= len(c):
        return None                         # not enough future
    leg_out_net = c[be + LEG_CANDLES] - c[be]

    return (
        classify_move(leg_in_net,  avg_atr),
        classify_move(leg_out_net, avg_atr),
        round(leg_in_net,  3),
        round(leg_out_net, 3),
    )


## 6. Walkthrough — Scenario A
Pull Scenario A's base candles out of the labeled fixtures, find their positions in the dataset, and call `measure_legs` directly. The output shows every number the classifier sees: ATR window, both leg windows, both nets, both directions, and the resulting formation.


In [22]:
scenario = "A_demand_RBR"
rows = labeled[labeled["scenario"] == scenario]

base_rows  = rows[rows["note"].str.startswith("BASE")]
base_start = df.index.get_loc(base_rows.index[0])
base_end   = df.index.get_loc(base_rows.index[-1])

result = measure_legs(base_start, base_end)
if result is None:
    print("Not enough candles to measure legs — skipped")
else:
    lin_dir, lout_dir, lin_net, lout_net = result
    formation = FORMATION_MAP.get((lin_dir, lout_dir), "unknown")
    avg_atr   = atr[base_start : base_end + 1].mean()
    threshold = LEG_ATR_MIN * avg_atr

    print(f"Scenario A — base candles: {base_start} → {base_end}")
    print(f"  Leg-in  window : {base_start - LEG_CANDLES} → {base_start - 1}")
    print(f"  Leg-out window : {base_end + 1} → {base_end + LEG_CANDLES}")
    print()
    print(f"  avg ATR        : {avg_atr:.3f}")
    print(f"  threshold      : {threshold:.3f}  (= {LEG_ATR_MIN}x ATR)")
    print(f"  leg-in  net    : {lin_net:+.3f}  → {lin_dir}")
    print(f"  leg-out net    : {lout_net:+.3f}  → {lout_dir}")
    print(f"  formation      : {formation}")


Scenario A — base candles: 19 → 21
  Leg-in  window : 16 → 18
  Leg-out window : 22 → 24

  avg ATR        : 1.587
  threshold      : 2.381  (= 1.5x ATR)
  leg-in  net    : +5.400  → up
  leg-out net    : +6.000  → up
  formation      : ('RBR', 'demand')


## 7. Full pipeline — every cluster, every scenario
Run the three stages in order:
1. Find raw clusters
2. Filter by `cluster_ok`
3. Measure legs and look up the formation

We collect every successful formation into a dict. NB06 will then add the departure filter on top, but already at this stage we should see all 9 scenario formations.


In [23]:
formations = []

for bs, be in find_base_clusters():
    if not cluster_ok(bs, be):
        continue
    result = measure_legs(bs, be)
    if result is None:
        continue
    lin_dir, lout_dir, lin_net, lout_net = result
    pair = FORMATION_MAP.get((lin_dir, lout_dir))
    if pair is None:
        continue
    formation, zone_type = pair
    scen = labeled["scenario"].iloc[bs]
    formations.append(dict(
        bs=bs, be=be,
        formation=formation, zone_type=zone_type,
        base_high=h[bs : be + 1].max(),
        base_low=l[bs : be + 1].min(),
        lin_dir=lin_dir, lout_dir=lout_dir,
        lin_net=lin_net, lout_net=lout_net,
        scenario=scen,
    ))

print(f"{len(formations)} formations detected (no departure filter yet)\n")
for f in formations:
    print(f"  {f['zone_type']:6s} {f['formation']}  {f['scenario']:<22s}  "
          f"leg-in={f['lin_net']:+.2f}  leg-out={f['lout_net']:+.2f}")


9 formations detected (no departure filter yet)

  demand RBR  A_demand_RBR            leg-in=+5.40  leg-out=+6.00
  supply RBD  A_demand_RBR            leg-in=+4.50  leg-out=-3.80
  supply DBD  B_supply_DBD            leg-in=-3.80  leg-out=-5.20
  demand RBR  D_wide_base             leg-in=+8.40  leg-out=+5.31
  demand RBR  E_doji_base             leg-in=+5.40  leg-out=+4.70
  demand RBR  E_doji_base             leg-in=+4.40  leg-out=+5.20
  demand DBR  H_DBR                   leg-in=-4.90  leg-out=+7.60
  demand RBR  H_DBR                   leg-in=+4.50  leg-out=+5.20
  supply RBD  I_RBD                   leg-in=+5.60  leg-out=-4.60


## 8. Visualization
Each detected formation is overlaid on the candle chart as a shaded box, color-coded by zone type:
- **Green** — demand zones (RBR, DBR)
- **Red** — supply zones (DBD, RBD)

The formation tag is annotated at the top-left of each box.


In [24]:
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "notebook_connected"

COLOR_BULL = "#26a69a"
COLOR_BEAR = "#ef5350"
BG         = "#131722"
GRID       = "#1e222d"
FILL       = {"demand": "rgba(38,166,154,0.18)", "supply": "rgba(239,83,80,0.18)"}
EDGE       = {"demand": "#26a69a",               "supply": "#ef5350"}

fig = go.Figure()

fig.add_trace(go.Candlestick(
    x=df.index,
    open=df["open"], high=df["high"],
    low=df["low"],   close=df["close"],
    increasing_line_color=COLOR_BULL,
    decreasing_line_color=COLOR_BEAR,
    name="Price",
))

for f in formations:
    x0 = df.index[f["bs"]]
    x1 = df.index[f["be"]]
    fig.add_vrect(
        x0=x0, x1=x1,
        fillcolor=FILL[f["zone_type"]],
        line=dict(color=EDGE[f["zone_type"]], width=1, dash="dot"),
        annotation_text=f["formation"],
        annotation_font=dict(size=9, color=EDGE[f["zone_type"]]),
        annotation_position="top left",
    )

fig.update_layout(
    title=f"Legs & Formation — {len(formations)} formations",
    height=560,
    plot_bgcolor=BG, paper_bgcolor=BG,
    font_color="white",
    xaxis_rangeslider_visible=False,
    hovermode="x unified",
    xaxis=dict(gridcolor=GRID, showgrid=True, zeroline=False),
    yaxis=dict(gridcolor=GRID, showgrid=True, zeroline=False, title="Price"),
)

fig.show()
